# Experiment 7 — Cross-product stability

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** For each policy, fit a separate linear model per vehicle and
compare:

- the coefficient patterns across vehicles (should be stable if the policy
  is truly margin-normalized)
- the train-on-$i$, test-on-$j$ transfer MAE matrix (should have a low
  mean regardless of $i \ne j$)

In [ ]:
def fit_per_vehicle(policy):
    feat, mdl, alpha = SELECTED[policy]
    vehicles = sorted(turns['vehicle'].dropna().unique())
    models = {}
    for v in vehicles:
        sub = turns[turns['vehicle'] == v]
        meta, X, y, cols = nu.build_design_matrix(sub, policy, feat)
        if len(X) < 30: continue
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler().fit(X)
        from sklearn.linear_model import Ridge, LinearRegression
        y_logit = nu.logit_clip(y)
        if mdl == 'OLS':
            reg = LinearRegression().fit(scaler.transform(X), y_logit)
        else:
            a = float(alpha) if np.isfinite(alpha) else 1.0
            reg = Ridge(alpha=a).fit(scaler.transform(X), y_logit)
        models[v] = {'scaler': scaler, 'reg': reg, 'cols': cols, 'X': X, 'y': y}
    return models

# Build per-vehicle models for all policies, aggregate coefficients
coef_records = []
for p in POLICIES:
    models = fit_per_vehicle(p)
    for v, info in models.items():
        for c, beta in zip(info['cols'], info['reg'].coef_):
            coef_records.append({'policy': p, 'vehicle': v, 'term': c, 'coef': beta})

coef_by_veh = pd.DataFrame(coef_records)
# Show coefficient stability for one policy (take first)
p0 = POLICIES[0]
print(f'Per-vehicle coefficients for {p0}:')
coef_by_veh[coef_by_veh.policy == p0].pivot(index='term', columns='vehicle', values='coef').round(3)

### Train-on-$i$, test-on-$j$ transfer matrix

For a chosen policy, fit on each vehicle and predict on every other, reporting
the mean absolute error on the normalized scale.

In [ ]:
from sklearn.metrics import mean_absolute_error

def transfer_matrix(policy):
    feat, mdl, alpha = SELECTED[policy]
    models = fit_per_vehicle(policy)
    vs = sorted(models.keys())
    mat = pd.DataFrame(index=vs, columns=vs, dtype=float)
    for vi in vs:
        src = models[vi]
        for vj in vs:
            tgt_X = models[vj]['X']
            tgt_y = models[vj]['y']
            X_scaled = src['scaler'].transform(tgt_X)
            yhat_logit = src['reg'].predict(X_scaled)
            yhat = nu.sigmoid(yhat_logit)
            mat.loc[vi, vj] = mean_absolute_error(tgt_y, yhat)
    return mat

# Show transfer matrix for first two policies
for p in POLICIES[:2]:
    print(f'\n=== {p} transfer matrix (rows = train vehicle, cols = test vehicle) ===')
    print(transfer_matrix(p).round(4))